# StarVector on Google Colab — Image2SVG

This notebook runs **StarVector-1B** and **StarVector-8B** image-to-SVG inference on Colab.

**GPU guide**
- `starvector-1b-im2svg` — needs ~4 GB VRAM. Runs on **free T4**.
- `starvector-8b-im2svg` — needs ~16–18 GB VRAM in fp16. Use **Colab Pro+ A100 (40 GB)** or load in 4-bit on smaller GPUs.

Set runtime: `Runtime → Change runtime type → GPU`.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install StarVector

In [ ]:
# Clone repo
!git clone https://github.com/joanrod/star-vector.git
%cd star-vector

# StarVector pins torch==2.5.1 + flash_attn==2.7.3 which often fail to build on
# fresh Colab. Install the heavy deps explicitly first, then install the
# package without re-running its dependency resolver.

# 1) Torch matching the pinned version. Skip this cell if you want to keep
#    Colab's preinstalled torch (then flash-attn must match that torch).
!pip install -q "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu121

# 2) flash-attn — use a prebuilt wheel and --no-build-isolation so it sees torch.
!pip install -q packaging ninja wheel
!pip install -q flash-attn==2.7.3 --no-build-isolation

# 3) taming-transformers from GitHub (the PyPI version is broken).
!pip install -q git+https://github.com/CompVis/taming-transformers.git

# 4) OpenAI CLIP from GitHub (the `clip-openai` pin is sometimes unreachable).
!pip install -q git+https://github.com/openai/CLIP.git || true

# 5) Install StarVector without re-resolving deps.
!pip install -q -e . --no-build-isolation --no-deps

# 6) Remaining lightweight deps.
!pip install -q transformers==4.49.0 tokenizers==0.21.1 sentencepiece==0.2.0 \
    accelerate pydantic==2.10 "numpy<2.0.0" scikit-learn==1.2.2 \
    svgpathtools==1.6.1 cairosvg beautifulsoup4 webcolors omegaconf \
    open-clip-torch datasets scikit-image fairscale lxml \
    sentence-transformers reportlab svglib Pillow protobuf openai \
    "markdown2[all]" httpx==0.24.0 requests uvicorn fastapi tqdm seaborn==0.12.2

# 7) For 4-bit loading of the 8B model on smaller GPUs.
!pip install -q bitsandbytes

In [ ]:
# Restart the runtime after the install above — torch was reinstalled.
# After this cell runs, the kernel will restart. Then start running again
# from cell "Check GPU" downward (the install cell does NOT need to run again).
import os
os.kill(os.getpid(), 9)

## 3. (Optional) HuggingFace login
Only needed if you hit a gated repo or want higher download throughput.

In [ ]:
# from huggingface_hub import login
# login(token='hf_...')

## 4. Shared helper — run inference and display result

We import `StarVectorForCausalLM` and a small util to rasterize the generated SVG so we can view it next to the input.

In [ ]:
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM
from starvector.data.util import process_and_rasterize_svg

def load_starvector(model_name: str, load_in_4bit: bool = False):
    kwargs = dict(trust_remote_code=True)
    if load_in_4bit:
        from transformers import BitsAndBytesConfig
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4',
        )
        kwargs['device_map'] = 'auto'
    else:
        kwargs['torch_dtype'] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    if not load_in_4bit:
        model.cuda()
    model.eval()
    return model

def image_to_svg(model, image_path: str, max_length: int = 4000):
    processor = model.model.processor
    image_pil = Image.open(image_path).convert('RGB')
    image = processor(image_pil, return_tensors='pt')['pixel_values'].cuda()
    if image.shape[0] != 1:
        image = image.squeeze(0)
    with torch.no_grad():
        raw_svg = model.generate_im2svg({'image': image}, max_length=max_length)[0]
    svg, raster = process_and_rasterize_svg(raw_svg)
    return svg, raster, image_pil

def show_pair(original, generated, title=''):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(original); ax[0].set_title('input'); ax[0].axis('off')
    ax[1].imshow(generated); ax[1].set_title('generated SVG'); ax[1].axis('off')
    fig.suptitle(title)
    plt.show()

## 5. StarVector-1B (fits on free Colab T4)

In [ ]:
model_1b = load_starvector('starvector/starvector-1b-im2svg')

In [ ]:
svg, raster, original = image_to_svg(model_1b, 'assets/examples/sample-0.png', max_length=1000)
show_pair(original, raster, title='StarVector-1B')
print(svg[:500], '...')

In [ ]:
with open('output_1b.svg', 'w') as f:
    f.write(svg)
from google.colab import files
files.download('output_1b.svg')

### Free the 1B before loading the 8B

In [ ]:
import gc
del model_1b
gc.collect(); torch.cuda.empty_cache()

## 6. StarVector-8B

Pick **one** of the two cells below depending on your GPU:
- **A100 / H100 / L4 24GB** → fp16 path.
- **T4 / V100 16GB** → 4-bit path (slower, slight quality drop).

In [ ]:
# Path A: fp16 (A100 / L4 24GB)
model_8b = load_starvector('starvector/starvector-8b-im2svg', load_in_4bit=False)

In [ ]:
# Path B: 4-bit (T4 / V100 16GB)
# model_8b = load_starvector('starvector/starvector-8b-im2svg', load_in_4bit=True)

In [ ]:
svg, raster, original = image_to_svg(model_8b, 'assets/examples/sample-0.png', max_length=4000)
show_pair(original, raster, title='StarVector-8B')
with open('output_8b.svg', 'w') as f:
    f.write(svg)
from google.colab import files
files.download('output_8b.svg')

## 7. Try your own image
Upload an icon / logo / diagram (PNG with clean shapes — StarVector is not trained on photos).

In [ ]:
from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))
svg, raster, original = image_to_svg(model_8b, path, max_length=4000)
show_pair(original, raster, title=path)